# 📋 Introduction to CrossTabs (Cross-Tabulation)

---

## What is a Cross-Tabulation?

A **cross-tabulation** (or **crosstab**) is a table that shows the **frequency or relationship between two (or more) categorical variables** simultaneously.

It answers questions like:
- "How many Toyota sedans vs hatchbacks vs wagons are in this dataset?"
- "What percentage of convertibles are made by Volkswagen?"
- "What is the average weight of each body style per manufacturer?"

**Familiar concept:** In Excel, this is similar to a PivotTable.

---

## `pd.crosstab()` vs Alternatives

The same result can be achieved three ways in pandas:

| Method | Syntax | Best For |
|---|---|---|
| `pd.crosstab()` | `pd.crosstab(index, columns)` | Quick frequency tables, normalisation |
| `groupby + unstack` | `df.groupby([...])['col'].count().unstack()` | When you're already grouping |
| `pivot_table` | `df.pivot_table(aggfunc=len)` | When you need custom aggregation |

---

## What You Will Learn

| Section | Topics |
|---|---|
| **Part 1: Setup** | Load and explore the car dataset |
| **Part 2: Basic Crosstab** | Frequency table, groupby and pivot_table alternatives |
| **Part 3: Margins** | Adding row and column subtotals |
| **Part 4: Aggregation** | Mean, sum, or any function on a value column |
| **Part 5: Normalisation** | Proportions — all, by column, by row |
| **Part 6: Multi-level Grouping** | Multiple index/column variables |
| **Part 7: Visualisation** | Heatmap of a crosstab using seaborn |

---

# Part 1: Setup — Load the Cars Dataset

---

We use an **automobile dataset** containing specifications for various car models.

We focus on **8 manufacturers**: Toyota, Nissan, Mazda, Honda, Mitsubishi, Subaru, Volkswagen, and Volvo.

**Key columns used:**

| Column | Description | Values |
|---|---|---|
| `make` | Car manufacturer | toyota, volkswagen, ... |
| `body_style` | Type of car body | sedan, hatchback, wagon, convertible, hardtop |
| `drive_wheels` | Wheel drive type | fwd, rwd, 4wd |
| `num_doors` | Number of doors | two, four |
| `curb_weight` | Car weight in lbs | numeric |

In [ ]:
# Suppress deprecation and future warnings to keep output clean
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Import pandas for data manipulation and crosstab
import pandas as pd

# Import seaborn for heatmap visualisation (Part 7)
import seaborn as sns

# Import matplotlib for plot display
import matplotlib.pyplot as plt

# Import numpy for numerical operations
import numpy as np

# Import StringIO so we can build the dataset in memory (self-contained notebook)
# In real usage: df_raw = pd.read_csv('../Data/cars.csv', header=None, names=headers)
from io import StringIO

# Define all column names (the original CSV has no header row)
# These column names come from the UCI Auto dataset specification
headers = ["symboling", "normalized_losses", "make", "fuel_type", "aspiration",
           "num_doors", "body_style", "drive_wheels", "engine_location",
           "wheel_base", "length", "width", "height", "curb_weight",
           "engine_type", "num_cylinders", "engine_size", "fuel_system",
           "bore", "stroke", "compression_ratio", "horsepower", "peak_rpm",
           "city_mpg", "highway_mpg", "price"]

# Build the cars dataset in memory
cars_csv = """3,?,alfa-romero,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495
1,?,toyota,gas,std,four,sedan,fwd,front,95.7,166.3,64.1,51.4,2190,ohc,four,98,2bbl,3.19,3.03,9.0,70,4800,30,37,6229
2,?,toyota,gas,std,four,sedan,fwd,front,95.7,170.7,64.1,56.6,2240,ohc,four,98,2bbl,3.19,3.03,9.0,70,4800,30,37,6918
1,?,toyota,gas,std,two,hatchback,fwd,front,95.7,161.1,63.9,52.0,2220,ohc,four,98,2bbl,3.19,3.03,9.0,70,4800,29,37,6488
1,?,toyota,gas,std,four,hatchback,fwd,front,95.7,161.1,63.9,52.0,2224,ohc,four,98,2bbl,3.19,3.03,9.0,70,4800,29,37,7898
2,?,toyota,gas,std,two,hardtop,fwd,front,95.7,163.7,63.9,52.0,2236,ohc,four,98,2bbl,3.19,3.03,9.0,70,4800,28,35,8778
1,?,toyota,gas,std,four,sedan,fwd,front,95.7,169.7,63.9,54.5,2264,ohc,four,98,2bbl,3.19,3.03,9.0,70,4800,27,34,9988
2,?,volkswagen,gas,std,two,sedan,fwd,front,97.3,171.7,65.5,55.7,2064,ohc,four,109,mpfi,3.19,3.40,8.5,88,5500,24,30,11595
2,?,volkswagen,gas,std,four,sedan,fwd,front,97.3,171.7,65.5,55.7,2136,ohc,four,109,mpfi,3.19,3.40,8.5,88,5500,24,30,12975
2,?,volkswagen,gas,std,two,hatchback,fwd,front,97.3,170.7,65.5,55.7,2088,ohc,four,109,mpfi,3.19,3.40,8.5,88,5500,24,30,12495
2,?,volkswagen,gas,std,two,convertible,fwd,front,97.3,166.5,65.5,52.3,2209,ohc,four,109,mpfi,3.19,3.40,8.5,88,5500,25,32,18420
1,?,honda,gas,std,four,sedan,fwd,front,86.6,144.6,63.9,50.8,1874,ohc,four,92,1bbl,2.91,3.41,9.6,58,4800,33,38,5151
1,?,honda,gas,std,two,hatchback,fwd,front,86.6,144.6,63.9,50.8,1835,ohc,four,92,1bbl,2.91,3.41,9.6,58,4800,33,38,5572
1,?,mitsubishi,gas,std,two,sedan,fwd,front,93.7,157.3,64.4,50.8,2128,sohc,four,122,2bbl,3.35,3.46,8.5,88,5000,25,32,7378
1,?,mitsubishi,gas,std,two,hatchback,fwd,front,93.7,157.3,64.4,50.8,2128,sohc,four,122,2bbl,3.35,3.46,8.5,88,5000,25,32,7898
2,?,mitsubishi,gas,std,two,hatchback,fwd,front,93.7,157.3,64.4,50.8,2128,sohc,four,122,2bbl,3.35,3.46,8.5,88,5000,25,32,8358
1,?,subaru,gas,std,four,sedan,fwd,front,86.6,144.6,63.9,54.5,1985,ohcf,four,97,2bbl,3.62,2.36,9.0,69,4900,31,36,7375
2,164,volvo,gas,std,four,sedan,rwd,front,104.3,188.8,67.2,56.2,2737,ohc,four,141,mpfi,3.78,3.15,9.5,114,5400,23,28,12940
2,164,volvo,gas,std,four,sedan,rwd,front,104.3,188.8,67.2,56.2,2824,ohc,four,141,mpfi,3.78,3.15,9.5,114,5400,23,28,13415
0,?,volvo,gas,std,four,wagon,rwd,front,104.3,188.8,67.2,57.3,2962,ohc,four,141,mpfi,3.78,3.15,9.5,114,5400,23,28,15445"""

# Read the CSV string with no header row — assign headers manually
# na_values='?': treat '?' as NaN (the dataset uses '?' for missing data)
df_raw = pd.read_csv(StringIO(cars_csv), header=None, names=headers, na_values='?')

# Define the 8 manufacturers we want to analyse
models = ["toyota", "nissan", "mazda", "honda", "mitsubishi", "subaru", "volkswagen", "volvo"]

# Filter: keep only rows where 'make' is in our target list
# .copy(): make an independent copy so we don't accidentally modify df_raw
df = df_raw[df_raw['make'].isin(models)].copy()

print(f"Total rows after filtering: {len(df)}")
print(f"Manufacturers: {sorted(df['make'].unique().tolist())}")
print(f"Body styles:   {sorted(df['body_style'].unique().tolist())}")
print()
print("Sample rows:")
print(df[['make', 'body_style', 'drive_wheels', 'num_doors', 'curb_weight']].head(8).to_string())

---

# Part 2: Basic Cross-Tabulation

---

## 2.1 `pd.crosstab()` — The Core Function

```python
pd.crosstab(index, columns)
```

- **`index`**: what goes on the **rows** (e.g., manufacturer name)
- **`columns`**: what goes on the **columns** (e.g., body style)
- **Result**: a table where each cell contains the **count** of rows matching that row-column combination

pandas does all the counting automatically.

In [ ]:
# ──────────────────────────────────────────────────────
# Basic crosstab: count of each body_style per make
# ──────────────────────────────────────────────────────
# index=df['make']: each unique value in 'make' becomes a ROW
# columns=df['body_style']: each unique value in 'body_style' becomes a COLUMN
# pandas counts how many rows have each (make, body_style) combination

print("=== How many cars of each body style does each manufacturer make? ===")
crosstab_basic = pd.crosstab(df['make'], df['body_style'])
print(crosstab_basic)
print()
print("Reading the table:")
print("  → Volkswagen makes 1 convertible, 1 hatchback, 1 sedan = 3 total")
print("  → Volvo makes 2 sedans, 1 wagon = 3 total")
print("  → 0 means that manufacturer makes NO cars of that body style")

In [ ]:
# ──────────────────────────────────────────────────────
# Alternative 1: groupby + unstack — same result
# ──────────────────────────────────────────────────────
# groupby(['make', 'body_style']): create groups for every combination
# ['body_style'].count(): count how many rows in each group
# .unstack(): pivot the inner groupby level (body_style) to become columns
# .fillna(0): replace NaN (no cars of that type) with 0

print("=== Same result via groupby + unstack ===")
result_groupby = (df.groupby(['make', 'body_style'])['body_style']
                    .count()
                    .unstack()
                    .fillna(0)
                    .astype(int))
print(result_groupby)

print()
# Verify both methods produce identical results
print("Crosstab vs groupby+unstack identical:", crosstab_basic.equals(result_groupby))

In [ ]:
# ──────────────────────────────────────────────────────
# Alternative 2: pivot_table — same result
# ──────────────────────────────────────────────────────
# index: row dimension (make)
# columns: column dimension (body_style)
# aggfunc={'body_style': len}: count the number of rows using len()
# fill_value=0: replace NaN with 0

print("=== Same result via pivot_table ===")
result_pivot = df.pivot_table(
    index=df['make'],
    columns=df['body_style'],
    aggfunc={'body_style': len},  # 'len' counts the number of matching rows
    fill_value=0
)
print(result_pivot)
print()
print("Summary: pd.crosstab() is the most concise for frequency tables.")
print("pivot_table is better when you need a custom aggregation function.")

---

# Part 3: Adding Subtotals — `margins`

---

## 3.1 The `margins` Parameter

Setting `margins=True` adds a **"Total"** row and column to the crosstab — summing across each row and each column.

This is equivalent to the **grand total** row/column in an Excel PivotTable.

```python
pd.crosstab(index, columns, margins=True, margins_name='Total')
```

- `margins=True`: add the subtotal row/column
- `margins_name='Total'`: the label for the subtotal (default is `'All'`)

In [ ]:
# ──────────────────────────────────────────────────────
# Crosstab with margins (subtotals)
# ──────────────────────────────────────────────────────
# index=df['make']: manufacturers as rows
# columns=df['num_doors']: door count (two/four) as columns
# margins=True: add a Total row AND column
# margins_name='Total': label for the subtotal row/column (default is 'All')

print("=== Crosstab with row and column subtotals ===")
crosstab_margins = pd.crosstab(
    index=df['make'],
    columns=df['num_doors'],
    margins=True,
    margins_name='Total'       # 'Total' appears as the label for the subtotal
)
print(crosstab_margins)

print()
print("Reading the margins:")
print("  → 'Total' column = sum of all door types for each manufacturer (row totals)")
print("  → 'Total' row    = sum of all manufacturers for each door type (column totals)")
print("  → Bottom-right cell = grand total of ALL rows in the dataset")

---

# Part 4: Aggregation — Computing Values Instead of Counts

---

## 4.1 `values` and `aggfunc` Parameters

By default, `pd.crosstab()` counts the number of rows per combination. But you can compute any aggregation (mean, sum, max, etc.) on a numeric column:

```python
pd.crosstab(
    index,
    columns,
    values=df['numeric_column'],  # which column to aggregate
    aggfunc='mean'               # what function to apply
)
```

**`aggfunc` accepts:**
- String shortcuts: `'mean'`, `'sum'`, `'min'`, `'max'`, `'count'`, `'std'`
- NumPy functions: `np.mean`, `np.sum`
- Custom functions: any function that accepts a Series and returns a scalar

In [ ]:
# ──────────────────────────────────────────────────────
# Crosstab with aggregation: mean curb_weight per make × body_style
# ──────────────────────────────────────────────────────
# values=df['curb_weight']: aggregate the curb_weight column
# aggfunc='mean': compute the average curb_weight for each cell
# .round(0): round to nearest integer for cleaner display

print("=== Mean curb weight (lbs) per manufacturer and body style ===")
crosstab_agg = pd.crosstab(
    index=df['make'],
    columns=df['body_style'],
    values=df['curb_weight'],     # the numeric column to aggregate
    aggfunc='mean'               # compute the mean for each cell
).round(0)
print(crosstab_agg)

print()
print("Reading this table:")
print("  → NaN = no cars of that body style for that manufacturer")
print("  → Numbers = average weight in lbs for that combination")

print()
print("=== Same with sum instead of mean ===")
# aggfunc='sum': total curb_weight for each cell
crosstab_sum = pd.crosstab(
    index=df['make'],
    columns=df['body_style'],
    values=df['curb_weight'],
    aggfunc='sum'
).round(0)
print(crosstab_sum)

---

# Part 5: Normalisation — Working with Proportions

---

## 5.1 The `normalize` Parameter

Instead of raw counts, `normalize` converts the crosstab to **proportions**:

| `normalize` value | What it computes |
|---|---|
| `True` or `'all'` | Each cell as a % of the **grand total** |
| `'columns'` | Each cell as a % of its **column total** |
| `'index'` | Each cell as a % of its **row total** |

All proportions in a normalised crosstab sum to 1.0 (in the relevant direction).

In [ ]:
# ──────────────────────────────────────────────────────
# Normalize='all' (or normalize=True)
# Each cell = count / grand_total
# ──────────────────────────────────────────────────────
# ALL cells together sum to 1.0
# Each cell tells you: 'what fraction of ALL cars in the dataset is this combination?'

print("=== normalize=True — each cell as fraction of GRAND TOTAL ===")
crosstab_norm_all = pd.crosstab(
    index=df['make'],
    columns=df['body_style'],
    normalize=True                # or normalize='all'
).round(3)
print(crosstab_norm_all)

print()
# Verify all cells sum to exactly 1.0 (minor floating point differences expected)
print(f"Sum of ALL cells: {crosstab_norm_all.values.sum():.4f} (should be 1.0)")
print()
print("Interpretation:")
print("  → If Volkswagen sedan shows 0.05, it means 5% of ALL cars are VW sedans")
print("  → The sum of the entire table = 1.0 (100% of the dataset)")

In [ ]:
# ──────────────────────────────────────────────────────
# Normalize='columns' — proportions within EACH column
# Each cell = count / column_total
# ──────────────────────────────────────────────────────
# Each COLUMN sums to 1.0 independently
# Answers: 'Of all convertibles, what fraction are made by Toyota?'

print("=== normalize='columns' — each cell as fraction of its COLUMN total ===")
crosstab_norm_cols = pd.crosstab(
    index=df['make'],
    columns=df['body_style'],
    normalize='columns'           # each column sums to 1.0
).round(3)
print(crosstab_norm_cols)

print()
# Verify: each column should sum to 1.0
print("Column sums (each should be 1.0 or NaN):")
print(crosstab_norm_cols.sum().round(3))
print()
print("Interpretation:")
print("  → Convertible column: Toyota=0.5, Volkswagen=0.5 means 50/50 split")
print("  → Sedan column: shows what fraction of all sedans each maker produces")

In [ ]:
# ──────────────────────────────────────────────────────
# Normalize='index' — proportions within EACH row
# Each cell = count / row_total
# ──────────────────────────────────────────────────────
# Each ROW sums to 1.0 independently
# Answers: 'Of all Mitsubishi cars, what fraction are hatchbacks?'

print("=== normalize='index' — each cell as fraction of its ROW total ===")
crosstab_norm_rows = pd.crosstab(
    index=df['make'],
    columns=df['body_style'],
    normalize='index'             # each row sums to 1.0
).round(3)
print(crosstab_norm_rows)

print()
# Verify: each row sums to 1.0
print("Row sums (each should be 1.0):")
print(crosstab_norm_rows.sum(axis=1).round(3))
print()
print("Interpretation:")
print("  → Mitsubishi row: hatchback=0.5, sedan=0.5 means 50% hatchbacks, 50% sedans")
print("  → Volvo row: sedan=0.667, wagon=0.333 means 2/3 sedans and 1/3 wagons")

---

## 5.2 Choosing the Right Normalisation

| Question | Use |
|---|---|
| "What fraction of ALL cars is this combination?" | `normalize=True` |
| "Among all cars of THIS body style, what fraction are from this maker?" | `normalize='columns'` |
| "Among all cars from THIS maker, what fraction are of this body style?" | `normalize='index'` |

**Example:**
- `normalize=True`: "Volvo sedans are 10% of the total dataset"
- `normalize='columns'` (sedan column): "Volvo makes 25% of all sedans in the dataset"
- `normalize='index'` (Volvo row): "67% of Volvo cars are sedans"

---

# Part 6: Multi-Level Grouping

---

## 6.1 Passing Lists to `index` and `columns`

You can pass **multiple columns** as lists to `index` or `columns`. Pandas creates a **multi-level (hierarchical) index** for the resulting table.

```python
pd.crosstab(
    index=[df['make'], df['num_doors']],        # rows: make × doors
    columns=[df['body_style'], df['drive_wheels']]  # cols: body × drive
)
```

This lets you explore **three or four dimensions** of data in a single table.

In [ ]:
# ──────────────────────────────────────────────────────
# Multi-column grouping — column dimension only
# ──────────────────────────────────────────────────────
# Pass a LIST of columns to create multi-level column headers
# cols: body_style AND drive_wheels together
# Result: columns are (body_style, drive_wheels) combinations

print("=== Multi-column grouping: body style × drive wheels ===")
# cols is a list of two Series — creates a 2-level column index
cols = [df['body_style'], df['drive_wheels']]
crosstab_multi_col = pd.crosstab(
    index=df['make'],    # rows: manufacturer name
    columns=cols         # columns: (body_style, drive_wheels) combinations
)
print(crosstab_multi_col)
print()
print("Reading: 'Volvo sedan rwd = 2' means Volvo makes 2 rear-wheel-drive sedans")

In [ ]:
# ──────────────────────────────────────────────────────
# Full multi-level: both rows AND columns are multi-level
# ──────────────────────────────────────────────────────
# idx: make AND num_doors together → 2-level row index
# cols: body_style AND drive_wheels together → 2-level column index

# List of Series for the row index (make × doors)
idx = [df['make'], df['num_doors']]

# List of Series for the column index (body_style × drive_wheels)
cols = [df['body_style'], df['drive_wheels']]

print("=== Full multi-level: make × doors (rows) vs body × drive (columns) ===")
crosstab_full_multi = pd.crosstab(
    index=idx,
    columns=cols,
    # Custom display names for the index and column levels
    # rownames: labels for the row index levels
    rownames=['Auto Manufacturer', 'Doors'],
    # colnames: labels for the column index levels
    colnames=['Body Style', 'Drive Type'],
    # dropna=False: include all combinations, even those with 0 cars
    # Without this, any (make, doors) combo with all-zero row would be hidden
    dropna=False
)
print(crosstab_full_multi)
print()
print("This table simultaneously shows 4 dimensions:")
print("  Row 1 (Manufacturer), Row 2 (Number of doors)")
print("  Col 1 (Body style),   Col 2 (Drive wheels type)")

---

# Part 7: Visualising Crosstabs with Seaborn Heatmaps

---

## 7.1 Why Heatmaps?

A **heatmap** maps numeric values to colours — darker or more intense colours indicate higher values. This lets you instantly spot patterns, clusters, and outliers in a crosstab.

**`sns.heatmap()` key parameters:**

| Parameter | Description |
|---|---|
| `data` | The DataFrame (our crosstab) |
| `cmap` | Colour palette (e.g., `'YlGnBu'`, `'Blues'`, `'RdYlGn'`) |
| `annot=True` | Show numeric value in each cell |
| `cbar=True` | Show the colour scale bar on the side |
| `fmt` | Number format for annotations (e.g., `'d'` for integer, `'.2f'` for 2 decimals) |
| `linewidths` | Width of grid lines between cells |

In [ ]:
# ──────────────────────────────────────────────────────
# Build the crosstab to visualise
# ──────────────────────────────────────────────────────
# Create a multi-level crosstab: make × num_doors vs body_style × drive_wheels
crosstab_viz = pd.crosstab(
    [df.make, df.num_doors],          # row index: manufacturer + door count
    [df.body_style, df.drive_wheels]  # columns: body style + drive type
)

print("Crosstab to be visualised:")
print(crosstab_viz)

In [ ]:
# ──────────────────────────────────────────────────────
# Heatmap visualisation using seaborn
# ──────────────────────────────────────────────────────

# Set the figure size before creating the heatmap
# figsize=(12, 6): 12 inches wide, 6 inches tall
plt.figure(figsize=(12, 6))

# sns.heatmap: creates a colour-coded grid from the crosstab DataFrame
# data=crosstab_viz: the crosstab table as the data source
# cmap='YlGnBu': colour palette from Yellow → Green → Blue (light = low, dark = high)
# annot=True: display the numeric count inside each cell
# cbar=True: show the colour bar (legend) on the right side
# fmt='d': format annotations as integers ('d' = decimal integer)
# linewidths=0.5: add thin lines between cells for readability
sns.heatmap(
    data=crosstab_viz,
    cmap='YlGnBu',         # colour palette: yellow=low, blue=high
    annot=True,             # show number in each cell
    cbar=True,              # show colour scale bar
    fmt='d',                # format as integer
    linewidths=0.5          # thin grid lines between cells
)

# Add title and axis labels for context
plt.title('Car Count Heatmap: Make × Doors vs Body Style × Drive Type', fontsize=13)
plt.xlabel('Body Style — Drive Type', fontsize=11)
plt.ylabel('Make — Number of Doors', fontsize=11)

# Rotate x-axis tick labels for readability when they're long
plt.xticks(rotation=30, ha='right')

# Adjust layout to prevent labels being cut off
plt.tight_layout()

# Display the plot
plt.show()

print("\nReading the heatmap:")
print("  → Darker cells = more cars of that combination")
print("  → Light/white cells = zero or very few cars")
print("  → Numbers inside each cell = exact count")

In [ ]:
# ──────────────────────────────────────────────────────
# Simpler heatmap: just make vs body_style (easier to read)
# ──────────────────────────────────────────────────────

# Simpler crosstab for a cleaner heatmap
crosstab_simple = pd.crosstab(df['make'], df['body_style'])

plt.figure(figsize=(9, 5))

# cmap='Blues': single-colour palette (white=0, dark blue=high)
# vmin=0: make sure 0 is at the white end of the scale
sns.heatmap(
    data=crosstab_simple,
    cmap='Blues',
    annot=True,
    cbar=True,
    fmt='d',
    linewidths=0.5,
    vmin=0               # ensure 0 maps to the lightest colour
)

plt.title('Car Count by Manufacturer and Body Style', fontsize=13)
plt.xlabel('Body Style', fontsize=11)
plt.ylabel('Manufacturer', fontsize=11)
plt.tight_layout()
plt.show()

---

# Part 8: Crosstab Quick-Reference Cheat Sheet

---

In [ ]:
# ──────────────────────────────────────────────────────
# Complete parameter reference — all in one place
# ──────────────────────────────────────────────────────

print("pd.crosstab() — Complete Parameter Reference")
print("=" * 60)

cheat_sheet = pd.DataFrame({
    'Parameter': [
        'index', 'columns', 'values', 'aggfunc',
        'margins', 'margins_name', 'normalize',
        'rownames', 'colnames', 'dropna'
    ],
    'Type': [
        'Series or list', 'Series or list', 'Series', 'str/func',
        'bool', 'str', "bool / 'index' / 'columns'",
        'list', 'list', 'bool'
    ],
    'Description': [
        'Row grouping variable(s)',
        'Column grouping variable(s)',
        'Numeric column to aggregate (requires aggfunc)',
        "Aggregation function: 'mean', 'sum', 'count', np.mean",
        'Add subtotal row and column',
        "Label for the subtotal (default: 'All')",
        "Normalise: True=all, 'index'=rows, 'columns'=cols",
        'Custom labels for row index levels',
        'Custom labels for column levels',
        'Include all categories even with count=0'
    ]
})

# Display as a formatted table
print(cheat_sheet.to_string(index=False))

---

# Summary

---

## Key Concepts at a Glance

### Core Usage

| Task | Code |
|---|---|
| Basic frequency table | `pd.crosstab(df['A'], df['B'])` |
| With subtotals | `pd.crosstab(index, cols, margins=True, margins_name='Total')` |
| Custom aggregation | `pd.crosstab(index, cols, values=df['C'], aggfunc='mean')` |
| As % of grand total | `pd.crosstab(index, cols, normalize=True)` |
| As % of column totals | `pd.crosstab(index, cols, normalize='columns')` |
| As % of row totals | `pd.crosstab(index, cols, normalize='index')` |
| Multi-level grouping | `pd.crosstab([df['A'], df['B']], [df['C'], df['D']])` |

### Three Ways to Get a Frequency Table

```python
# Method 1: crosstab (most concise)
pd.crosstab(df['make'], df['body_style'])

# Method 2: groupby + unstack
df.groupby(['make','body_style'])['body_style'].count().unstack().fillna(0)

# Method 3: pivot_table
df.pivot_table(index='make', columns='body_style', aggfunc={'body_style': len}, fill_value=0)
```

### When to Normalise

| Question | `normalize` value |
|---|---|
| What % of ALL data is this cell? | `True` |
| Of all X, what % come from each Y? | `'columns'` |
| Of all Y, what % are each X? | `'index'` |

---

## Self-Test Questions

1. What is the difference between `pd.crosstab()` and `df.pivot_table()`? When would you choose each?
2. What does `margins=True` add to the crosstab output? What does the bottom-right corner cell represent?
3. If `normalize='columns'` is used and the 'sedan' column shows Toyota=0.4, what does this mean?
4. If `normalize='index'` is used and the 'Mitsubishi' row shows hatchback=0.5, what does this mean?
5. What does `dropna=False` do in a multi-level crosstab?
6. How would you create a crosstab that shows the TOTAL price instead of counts?
7. In a seaborn heatmap, what does `annot=True` do? What does `cmap='YlGnBu'` mean?